In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from collections import defaultdict
import json

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

BUNDLE_DIR = Path.cwd().resolve()
ROOT_DIR = BUNDLE_DIR.parents[0]
RESULTS_DIR = BUNDLE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = ROOT_DIR / "data"
PROTEIN_DATA_DIR = DATA_DIR / "protein" / "aggregated_all"
CELL_LINEAGE_PATH = DATA_DIR / "cell_lineage.json"
CELL_TYPE_PATH = DATA_DIR / "2023-06-29_entropy_cell_key_V2.csv"
S3_PATH = PROTEIN_DATA_DIR / "s3.csv"

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))

def map_names(did):
    if   did == "P4a": return "Z3"
    elif did == "P4p": return "Z2"
    elif did == "P0a": return "AB"
    else: return did

In [2]:
# ── Lineage tree & terminal/intermediate enumeration ──
with CELL_LINEAGE_PATH.open('r', encoding='utf-8') as f:
    lineage_data = json.load(f)
terminal_nodes = []
intermediate_nodes = []
descendant_list_dict = defaultdict(list)

def dfs(node, parent, ancestors=[]):
    children = node.get("children", [])
    lookup_name = map_names(node["did"])
    if len(children) == 0:
        terminal_nodes.append(lookup_name)
        for ancestor in ancestors:
            descendant_list_dict[ancestor].append(lookup_name)
    else:
        intermediate_nodes.append(lookup_name)
        for child in children:
            dfs(child, node, ancestors + [lookup_name])

dfs(lineage_data, None)
print(f"Intermediate: {len(intermediate_nodes)}  |  Terminal: {len(terminal_nodes)}")

Intermediate: 1091  |  Terminal: 1092


In [3]:
# ── Assign cell types to terminal nodes ──
cell_type_df = pd.read_csv(CELL_TYPE_PATH)
cell_type_dict = {}
for node in terminal_nodes:
    cur = cell_type_df[cell_type_df['wormweb.lineage'] == node]
    if len(cur) == 0:
        print(f"Warning: No cell type found for {node}")
        continue
    types = cur["wormweb.type"].dropna().unique()
    if len(types) == 0:
        cell_type_dict[node] = "programmed_death"
        continue
    if len(types) > 1:
        print(f"Warning: Multiple cell types found for {node}: {types}")
    cell_type_dict[node] = types[0]

In [4]:
# ─── Experiment toggles ────────────────────────────────────────────────────
CELL_TYPE_MERGE   = None    # None (original) | "A" (Schema A) | "D" (Schema D)
EXCLUDE_DEAD      = False   # exclude programmed_death terminal cells from training
LINEAR_MODEL      = "l2"    # "l2" (ridge) | "l1" (lasso) | "elasticnet"
C_VALUE           = 1.0     # inverse regularisation strength (smaller = stronger)
N_CV_FOLDS        = 5

# Output directory
_schema = f"_schema{CELL_TYPE_MERGE}" if CELL_TYPE_MERGE else ""
_dead  = "_no_dead" if EXCLUDE_DEAD else ""
RUN_NAME = f"linear_{LINEAR_MODEL}{_schema}{_dead}"
RUN_DIR = RESULTS_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"CELL_TYPE_MERGE = {CELL_TYPE_MERGE}")
print(f"EXCLUDE_DEAD    = {EXCLUDE_DEAD}")
print(f"LINEAR_MODEL    = {LINEAR_MODEL}  C={C_VALUE}")
print(f"Output          -> {RUN_DIR}")

CELL_TYPE_MERGE = None
EXCLUDE_DEAD    = False
LINEAR_MODEL    = l2  C=1.0
Output          -> /home/bingran/code/embryogenesis/expression_embedding/results/linear_l2


In [5]:
# ─── Cell type merge (Schema A or D) ───────────────────────────────────────
_original_other_nodes = [node for node, ct in cell_type_dict.items() if ct == "other"]
_DEATH_LABEL_HINT = "death"

if CELL_TYPE_MERGE == "A":
    MERGE_MAP = {
        "neuron": "Neuron", "muscle": "Muscle",
        "programmed_death": "Programmed death",
        "hypoderm": "Skin", "tail": "Skin", "epithelium": "Skin",
        "mesoderm": "Skin", "repro": "Skin",
        "sheath": "Glia + excretory", "socket": "Glia + excretory",
        "excretory": "Glia + excretory", "coelomocyte": "Glia + excretory",
        "gland": "Glia + excretory",
        "marginal": "Pharynx + rectal", "valve": "Pharynx + rectal",
        "rectal": "Pharynx + rectal", "intestine": "Pharynx + rectal",
    }
elif CELL_TYPE_MERGE == "D":
    MERGE_MAP = {
        "neuron": "neuron", "muscle": "muscle", "repro": "reproduction",
        "hypoderm": "epithelium", "epithelium": "epithelium",
        "sheath": "glial", "socket": "glial",
        "coelomocyte": "coelomocyte", "excretory": "excretory",
        "mesoderm": "mesoderm", "other": "other",
        "intestine": "alimentary", "valve": "alimentary",
        "marginal": "alimentary", "gland": "alimentary",
        "rectal": "alimentary",
        "tail": "programmed_death", "programmed_death": "programmed_death",
    }
    _DEATH_LABEL_HINT = "programmed_death"

if CELL_TYPE_MERGE is not None:
    n_remapped = 0
    for node in cell_type_dict:
        if cell_type_dict[node] in MERGE_MAP:
            cell_type_dict[node] = MERGE_MAP[cell_type_dict[node]]
            n_remapped += 1
    merged = [v for v in set(cell_type_dict.values()) if v != "other" and v is not None]
    print(f"Schema {CELL_TYPE_MERGE}: {n_remapped} remapped -> {len(merged)} classes")
else:
    print(f"Original: {len(set(cell_type_dict.values()))} cell types")

Original: 18 cell types


In [6]:
# ─── One-hot encoding ──────────────────────────────────────────────────────
cell_types = list(set(cell_type_dict.values()))
# Find death class label for downstream tracking
_DEATH_LABEL = None
for ct in cell_types:
    if ct is not None and _DEATH_LABEL_HINT in ct.lower():
        _DEATH_LABEL = ct
        break
if _DEATH_LABEL is None:
    _DEATH_LABEL = "programmed_death"

cell_types = [ct for ct in cell_types if ct != _DEATH_LABEL]
cell_types = sorted(cell_types, key=lambda x: sum(1 for v in cell_type_dict.values() if v == x), reverse=True)
cell_types.append(_DEATH_LABEL)
cell_type_to_int = {ct: i for i, ct in enumerate(cell_types)}

cell_type_one_hot = {}
for node, ct in cell_type_dict.items():
    oh = np.zeros(len(cell_types))
    oh[cell_type_to_int[ct]] = 1.0
    cell_type_one_hot[node] = oh

for node in intermediate_nodes:
    desc_oh = [cell_type_one_hot[d] for d in descendant_list_dict[node]]
    if len(desc_oh) == 0:
        cell_type_one_hot[node] = np.zeros(len(cell_types))
    else:
        summed = np.sum(desc_oh, axis=0)
        cell_type_one_hot[node] = summed / np.sum(summed)

print(f"Output classes: {len(cell_types)}")

Output classes: 18


In [7]:
# ─── Protein expression data ───────────────────────────────────────────────
protein_exp = pd.read_csv(S3_PATH, index_col=0).T.fillna(0)
protein_exp = protein_exp[~(protein_exp == 0).all(axis=1)]
protein_exp_zscore = protein_exp.apply(lambda x: (x - x.mean()) / x.std(), axis=1)

# Standardise features (columns) for linear model
X_full = protein_exp_zscore.values.T.astype(np.float64)
X_full = StandardScaler().fit_transform(X_full)

y_full = np.array([cell_type_one_hot[map_names(n)] for n in protein_exp_zscore.columns])
sample_names = np.array([map_names(n) for n in protein_exp_zscore.columns])

# Hard-label (terminal) / soft-label (intermediate) split
is_terminal = np.array([
    len(descendant_list_dict[map_names(n)]) == 0
    for n in protein_exp_zscore.columns
])

# Hard label for classification (argmax over soft distribution)
y_hard = y_full.argmax(axis=1)

print(f"X: {X_full.shape}  |  y: {y_full.shape}")
print(f"Terminal (hard): {is_terminal.sum()}  |  Intermediate (soft): {(~is_terminal).sum()}")

X: (1204, 210)  |  y: (1204, 18)
Terminal (hard): 494  |  Intermediate (soft): 710


In [8]:
# ─── Build training mask (exclusions) ──────────────────────────────────────
train_mask = np.ones(len(y_full), dtype=bool)

if EXCLUDE_DEAD:
    death_idx = cell_type_to_int.get(_DEATH_LABEL, -1)
    is_dead_terminal = is_terminal & (y_hard == death_idx)
    train_mask &= ~is_dead_terminal
    print(f"Dead terminal excluded: {is_dead_terminal.sum()}")

if CELL_TYPE_MERGE is not None:
    is_other_terminal = np.array([
        is_terminal[i] and sample_names[i] in _original_other_nodes
        for i in range(len(sample_names))
    ])
    train_mask &= ~is_other_terminal
    print(f"'other' cells excluded: {is_other_terminal.sum()}")

X = X_full[train_mask]
y_hard_train = y_hard[train_mask]
y_full_train = y_full[train_mask]
is_terminal_train = is_terminal[train_mask]

print(f"Training samples: {len(X)} / {len(X_full)}")
print(f"  Terminal: {is_terminal_train.sum()}  Intermediate: {(~is_terminal_train).sum()}")

Training samples: 1204 / 1204
  Terminal: 494  Intermediate: 710


In [9]:
# ─── Train & cross-validate ────────────────────────────────────────────────
# LogisticRegression — sklearn ≥1.8 uses l1_ratio instead of penalty:
#   l1_ratio=0 → L2 (ridge),  l1_ratio=1 → L1 (lasso),  0<l1_ratio<1 → elasticnet

if LINEAR_MODEL == 'l2':
    l1_ratio = 0.0
elif LINEAR_MODEL == 'l1':
    l1_ratio = 1.0
else:  # elasticnet
    l1_ratio = 0.5

model = LogisticRegression(
    C=C_VALUE,
    l1_ratio=l1_ratio,
    max_iter=5000,
    tol=1e-4,
    random_state=42,
)

# 5-fold stratified CV on training hard-label terminal cells
X_term = X[is_terminal_train]
y_term = y_hard_train[is_terminal_train]

cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
fold_accs = []
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_term, y_term)):
    X_tr, X_val = X_term[tr_idx], X_term[val_idx]
    y_tr, y_val = y_term[tr_idx], y_term[val_idx]
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    fold_accs.append(acc)
    print(f"  Fold {fold+1}/{N_CV_FOLDS}: val_acc={acc:.4f}")

cv_mean = np.mean(fold_accs)
cv_std  = np.std(fold_accs)
print(f"\nCV accuracy (terminal, hard-label): {cv_mean:.4f} +/- {cv_std:.4f}")
print(f"(The CV score is the honest estimate — the model will overfit on all data)")

# Fit on all training terminal data for full-data evaluation
model.fit(X_term, y_term)
print(f"Model coef shape: {model.coef_.shape}  (n_classes x n_features)")

  Fold 1/5: val_acc=0.7374
  Fold 2/5: val_acc=0.6869
  Fold 3/5: val_acc=0.7879
  Fold 4/5: val_acc=0.7273


  Fold 5/5: val_acc=0.7041

CV accuracy (terminal, hard-label): 0.7287 +/- 0.0345
(The CV score is the honest estimate — the model will overfit on all data)


Model coef shape: (16, 210)  (n_classes x n_features)


/home/bingran/miniconda3/envs/dev/lib/python3.13/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [10]:
# ─── Evaluate on full dataset ──────────────────────────────────────────────
probs_all = model.predict_proba(X_full)
preds_all = model.predict(X_full)

# Terminal (hard-label) accuracy
term_preds   = preds_all[is_terminal]
term_targets = y_hard[is_terminal]

term_acc = accuracy_score(term_targets, term_preds)
print(f"Terminal accuracy (hard-label): {term_acc:.4f}")

if EXCLUDE_DEAD:
    non_dead_mask = is_terminal & ~is_dead_terminal
    dead_mask = is_terminal & is_dead_terminal
    acc_nd = accuracy_score(y_hard[non_dead_mask], preds_all[non_dead_mask])
    acc_d  = accuracy_score(y_hard[dead_mask], preds_all[dead_mask])
    print(f"  Non-dead terminal: {acc_nd:.4f}")
    print(f"  Dead terminal (unseen): {acc_d:.4f}")

# Per-class breakdown
per_class = defaultdict(list)
for p, t in zip(term_preds, term_targets):
    per_class[cell_types[t]].append(p == t)

print(f"\n{'Class':30s}  {'Acc':>6}  {'n':>4}")
print("-" * 45)
for ct, vals in sorted(per_class.items(), key=lambda x: -np.mean(x[1])):
    print(f"  {ct:28s}  {np.mean(vals):6.3f}  {len(vals):4d}")

Terminal accuracy (hard-label): 1.0000

Class                              Acc     n
---------------------------------------------
  neuron                         1.000   167
  programmed_death               1.000    74
  sheath                         1.000    22
  socket                         1.000    18
  epithelium                     1.000    13
  muscle                         1.000   110
  other                          1.000     9
  marginal                       1.000     9
  hypoderm                       1.000    51
  excretory                      1.000     4
  rectal                         1.000     2
  tail                           1.000     2
  valve                          1.000     8
  intestine                      1.000     2
  mesoderm                       1.000     1
  repro                          1.000     2


In [11]:
# ─── Top features per class (from linear model coefficients) ──────────────
feature_names = np.array(protein_exp_zscore.index.tolist())

n_top = 10
print(f"Top {n_top} features per class (by |coefficient|):\n")
for i, class_idx in enumerate(model.classes_):
    ct = cell_types[class_idx]
    coefs = np.abs(model.coef_[i])
    top_idx = np.argsort(coefs)[::-1][:n_top]
    top_features = [f"{feature_names[j]} ({coefs[j]:.3f})" for j in top_idx]
    print(f"{ct:28s}: {', '.join(top_features[:5])}")

# Save feature rankings
importance_df = pd.DataFrame({
    'feature': feature_names,
    **{f'{cell_types[class_idx]}_coef': model.coef_[i]
       for i, class_idx in enumerate(model.classes_)}
})
importance_df.to_csv(RUN_DIR / "linear_coefficients.csv", index=False)
print(f"\nSaved coefficients -> {RUN_DIR / 'linear_coefficients.csv'}")

Top 10 features per class (by |coefficient|):

neuron                      : CND-1_SYS393 (0.862), HLH-14_SYS461 (0.842), NHR-25_RW10348 (0.748), NGN-1_SYS498 (0.707), ZAG-1_SYS102 (0.697)
hypoderm                    : NHR-25_RW10348 (0.791), CEH-45_SYS634 (0.650), CES-2_SYS443 (0.585), HAM-2_SYS553 (0.448), PHA-4_RW10425 (0.441)
muscle                      : M03D4.4_SYS453 (0.602), UNC-120_SYS167 (0.596), CEBP-1_SYS451 (0.569), CEH-34_SYS173 (0.546), F19F10.9_SYS423 (0.523)
repro                       : LSL-1_SYS419 (0.343), ALY-2_SYS721 (0.118), RCOR-1_SYS667 (0.081), HMG-20_SYS477 (0.080), F19F10.9_SYS423 (0.063)
intestine                   : ELT-2_SYS412 (0.153), NHR-232_SYS395 (0.132), DVE-1_SYS165 (0.123), ELT-2_RW10714 (0.116), ELT-7_SYS543 (0.110)
epithelium                  : PHP-3_SYS1013 (0.494), HBL-1_SYS641 (0.475), CEH-45_SYS634 (0.456), SWSN-7_SYS100 (0.410), CEH-27_SYS89 (0.407)
socket                      : HAM-2_SYS553 (0.736), ZTF-11_SYS99 (0.630), DUXL-1_SYS431 (0.5